# Pilot Dataset Collection

한국 손해배상 민사 판결문 50개 + 미국 tort/damages 판결문 50개 수집

- 한국: `lbox/lbox_open` (precedent_corpus)
- 미국: `harvard-lil/cold-cases` (streaming)
- 출력: `kr_cases.csv`, `us_cases.csv`

## 2. 설정

In [9]:
from pathlib import Path

# --- 수집 목표 ---
SAMPLE_SIZE = 50          # 각 jurisdiction별 수집 목표 수
SEED = 42

# --- 한국 데이터셋 ---
KR_DATASET   = "lbox/lbox_open"
KR_CONFIG    = "precedent_corpus"
KR_SPLIT     = "train"
KR_TEXT_COL  = "precedent"   # 판결문 전문 컬럼

# 1심/2심 필터: '상고'/'재항고' 언급 없는 것 = trial·appellate
KR_EXCLUDE_PATTERNS = ["상고이유", "재항고"]

# 손해배상 관련 키워드 (하나 이상 포함 시 수집)
KR_KEYWORDS = ["손해배상", "불법행위", "위자료", "과실상계"]

# --- 미국 데이터셋 ---
US_DATASET       = "harvard-lil/cold-cases"
US_SPLIT         = "train"
US_COURT_TYPES   = {"FD", "ST", "FA", "SA"}  # FD=연방지방, ST=주1심, FA=연방항소, SA=주항소
US_SCAN_LIMIT    = 200_000        # 스트리밍 탐색 최대 행 수 (안전장치)

# tort/damages 관련 키워드
US_NOS_KEYWORDS  = [             # nature_of_suit 필드 대상
    "tort", "personal injury", "negligence",
    "property damage", "product liability",
    "medical malpractice", "wrongful death",
]
US_TEXT_KEYWORDS = [             # 본문 대상 (NOS 없을 때 fallback)
    "damages", "negligence", "tort",
    "duty of care", "proximate cause",
]
US_MIN_TEXT_LEN  = 1_000         # 너무 짧은 판결문 제외 (글자 수)

# --- 출력 경로 ---
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KR_OUTPUT = OUTPUT_DIR / "kr_cases.csv"
US_OUTPUT = OUTPUT_DIR / "us_cases.csv"

print("설정 완료")
print(f"  목표 샘플 수 : {SAMPLE_SIZE}개/jurisdiction")
print(f"  출력 경로    : {OUTPUT_DIR.resolve()}")

설정 완료
  목표 샘플 수 : 50개/jurisdiction
  출력 경로    : C:\Users\sn031\OneDrive\바탕 화면\comparative_law_llm_pc\comparative-law-llm\outputs


## 3. 한국 판결문 수집

`lbox/lbox_open` 전체를 DataFrame으로 로드한 뒤,
손해배상 키워드 필터 → 1심/2심 필터 → 무작위 50개 샘플링

In [10]:
import pandas as pd
from datasets import load_dataset

print("한국 데이터셋 로딩 중...")
kr_raw = load_dataset(KR_DATASET, KR_CONFIG, split=KR_SPLIT)
kr_df  = kr_raw.to_pandas()
print(f"전체 행 수: {len(kr_df):,}")
print(f"컬럼: {list(kr_df.columns)}")

한국 데이터셋 로딩 중...
전체 행 수: 150,000
컬럼: ['id', 'precedent']


In [11]:
# 손해배상 키워드 필터
kw_pattern = "|".join(KR_KEYWORDS)
mask_kw    = kr_df[KR_TEXT_COL].str.contains(kw_pattern, na=False, regex=True)
kr_kw      = kr_df[mask_kw].copy()
print(f"키워드 필터 후: {len(kr_kw):,}건")

# 상고/재항고(대법원) 제외 → 1심·2심만 유지
excl_pattern = "|".join(KR_EXCLUDE_PATTERNS)
mask_excl    = kr_kw[KR_TEXT_COL].str.contains(excl_pattern, na=False, regex=True)
kr_trial     = kr_kw[~mask_excl].copy()
print(f"1·2심 필터 후: {len(kr_trial):,}건")

# 무작위 50개 샘플링
kr_sample = kr_trial.sample(n=min(SAMPLE_SIZE, len(kr_trial)), random_state=SEED).reset_index(drop=True)
print(f"최종 샘플: {len(kr_sample)}건")

키워드 필터 후: 21,980건
1·2심 필터 후: 14,778건
최종 샘플: 50건


In [12]:
# 공통 스키마로 정리
kr_out = pd.DataFrame({
    "case_id"      : kr_sample["id"].astype(str),
    "jurisdiction" : "KR",
    "case_name"    : kr_sample.get("case_name", pd.Series([""] * len(kr_sample))),
    "court_name"   : kr_sample.get("court_name", pd.Series([""] * len(kr_sample))),
    "court_level"  : kr_sample.get("court_level", pd.Series([""] * len(kr_sample))),
    "decision_date": kr_sample.get("decision_date", pd.Series([""] * len(kr_sample))),
    "raw_text"     : kr_sample[KR_TEXT_COL],
})

kr_out.to_csv(KR_OUTPUT, index=False, encoding="utf-8-sig")
print(f"저장 완료: {KR_OUTPUT}  ({len(kr_out)}건)")
kr_out[["case_id", "court_name", "court_level", "decision_date"]].head()

저장 완료: outputs\kr_cases.csv  (50건)


,case_id,court_name,court_level,decision_date
0,104947,,,
1,121915,,,
2,38911,,,
3,66659,,,
4,96329,,,


## 4. 미국 판결문 수집

`harvard-lil/cold-cases`를 스트리밍으로 탐색.
court_type 필터(FD/ST) → nature_of_suit 또는 본문 키워드 필터 → 50개 수집 시 중단

In [13]:
from datasets import load_dataset
from tqdm.auto import tqdm


def extract_opinion_text(row: dict) -> str:
    """opinions 리스트에서 본문 텍스트를 이어붙임."""
    opinions = row.get("opinions") or []
    if isinstance(opinions, list):
        return " ".join(op.get("opinion_text") or op.get("text") or "" for op in opinions if isinstance(op, dict))
    return ""


def matches_us_criteria(row: dict) -> bool:
    """court_type + 키워드 기준으로 수집 여부 판단."""
    court_type = (row.get("court_type") or "").upper()
    if court_type not in US_COURT_TYPES:
        return False

    nos = (row.get("nature_of_suit") or "").lower()
    if any(kw in nos for kw in US_NOS_KEYWORDS):
        return True

    # nature_of_suit 없으면 본문 키워드로 fallback
    text = extract_opinion_text(row).lower()
    return len(text) >= US_MIN_TEXT_LEN and any(kw in text for kw in US_TEXT_KEYWORDS)


print("미국 데이터셋 스트리밍 탐색 중 (시간이 다소 걸릴 수 있습니다)...")
us_ds = load_dataset(US_DATASET, split=US_SPLIT, streaming=True)

us_records = []
scanned    = 0

with tqdm(total=SAMPLE_SIZE, desc="US cases collected") as pbar:
    for row in us_ds:
        scanned += 1
        if scanned > US_SCAN_LIMIT:
            print(f"[경고] {US_SCAN_LIMIT:,}행 탐색 후 중단 (수집: {len(us_records)}건)")
            break

        if not matches_us_criteria(row):
            continue

        text = extract_opinion_text(row)
        if len(text) < US_MIN_TEXT_LEN:
            continue

        us_records.append({
            "case_id"      : str(row.get("id") or f"US_{scanned}"),
            "jurisdiction" : "US",
            "case_name"    : row.get("case_name") or row.get("name") or "",
            "court_name"   : row.get("court_full_name") or row.get("court") or "",
            "court_level"  : "trial" if row.get("court_type") in {"FD", "ST"} else "appellate",
            "decision_date": row.get("date_filed") or row.get("date") or "",
            "nature_of_suit": row.get("nature_of_suit") or "",
            "raw_text"     : text,
        })
        pbar.update(1)

        if len(us_records) >= SAMPLE_SIZE:
            break

print(f"\n탐색 행 수: {scanned:,} | 수집 완료: {len(us_records)}건")

미국 데이터셋 스트리밍 탐색 중 (시간이 다소 걸릴 수 있습니다)...


US cases collected: 100%|██████████| 50/50 [00:27<00:00,  1.79it/s]


탐색 행 수: 424 | 수집 완료: 50건


In [14]:
us_out = pd.DataFrame(us_records)

us_out.to_csv(US_OUTPUT, index=False, encoding="utf-8-sig")
print(f"저장 완료: {US_OUTPUT}  ({len(us_out)}건)")
us_out[["case_id", "court_name", "decision_date", "nature_of_suit"]].head()

저장 완료: outputs\us_cases.csv  (50건)


,case_id,court_name,decision_date,nature_of_suit
0,1742237,"District Court, S.D. Iowa",1996-05-15,
1,2333112,"District Court, D. Utah",2004-10-04,
2,2427282,"District Court, D. Kansas",2004-04-29,
3,1083174,Court of Criminal Appeals of Tennessee,1997-12-10,
4,2201444,Appellate Court of Illinois,1972-10-17,


## 5. 간단 확인

In [15]:
print("=" * 40)
print(f"KR 판결문: {len(kr_out)}건  →  {KR_OUTPUT}")
print(f"US 판결문: {len(us_out)}건  →  {US_OUTPUT}")
print("=" * 40)

print("\n[KR] 텍스트 길이 분포 (글자 수)")
print(kr_out["raw_text"].str.len().describe().astype(int))

print("\n[US] 텍스트 길이 분포 (글자 수)")
print(us_out["raw_text"].str.len().describe().astype(int))

KR 판결문: 50건  →  outputs\kr_cases.csv
US 판결문: 50건  →  outputs\us_cases.csv

[KR] 텍스트 길이 분포 (글자 수)
count       50
mean      5227
std       5515
min        200
25%       1570
50%       3458
75%       6067
max      25404
Name: raw_text, dtype: int64

[US] 텍스트 길이 분포 (글자 수)
count       50
mean     28203
std      16499
min       1590
25%      15869
50%      23872
75%      37725
max      66925
Name: raw_text, dtype: int64
